<a href="https://colab.research.google.com/github/Kohei-200/math/blob/main/adaBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## AdaBoost: a Boosting technique for classification


Reference:

[Ensemble Methods Foundations and Algorithms]((https://www.taylorfrancis.com/books/mono/10.1201/9781003587774/ensemble-methods-zhi-hua-zhou))


ByZhi-Hua Zhou

Boosting = loop {Train -> adjust distribution} -> combine outputs

Boosting reduces bias

### General Procedure
Input:

Sample Dist: **D**

Base learning algo: **L**

Number of learning rounds: **T**



**Procedure**:

\begin{align}
&\mathbf{D_1} = \mathbf{D} \tag{1} \\
&\text{for t = 1 ... T;} \tag{2}\\
&\mathbf{h_t} = \mathbf{L}(D_t) \tag{3. train} \\
&\mathbf{\epsilon}_t = \mathbf{P}_\mathbf{x_\tilde{D_t}}(h_t(\mathbf{x}) \neq f(\mathbf{x})) \tag{4. evaluate the error}\\
&\text{if }\mathbf{\epsilon}_t > 1/2 \text{: then break} \tag{5. early stopping}\\
&\alpha_t = \frac{1}{2}\ln(\frac{1 - \epsilon_t}{\epsilon_t}) \tag{6. accurate = high say} \\
&\mathbf{D_{t+1}}(\mathbf{x}) = \frac{\mathbf{D_t}(\mathbf{x})}{Z_t} × \begin{cases} \exp(-\alpha_t)\text{ if } h_t = f \\ \exp(\alpha_t)\text{ if } h_t \neq f\end{cases} \tag{7}\\
& = \frac{\mathbf{D_t}(\mathbf{x})\exp(-\alpha_t ⋅ f(\mathbf{x}) ⋅ h_t(\mathbf{x}))}{Z_t} \tag{7}\\
&end \tag{8}\\
&output = sign(H(\mathbf{x})) \text{ with } H = \sum_{t=1}^T \alpha_t h_t(\mathbf{x}) \tag{9}\\
\end{align}

## Code

In [111]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np

In [112]:
X, y = make_classification(
    n_samples = 500,
    n_features = 2,
    n_redundant = 0,
    random_state = 42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2)
y_train = np.where(y_train == 0, -1, 1) # AdaBoost takes -1/+1
y_test = np.where(y_test == 0, -1, 1) # AdaBoost takes -1/+1

In [113]:
D_1 = np.ones(X_train.shape[0])/X_train.shape[0] # uniform dist

In [114]:
D_1

array([0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
       0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
       0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
       0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
       0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
       0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
       0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
       0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
       0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
       0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
       0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
       0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
       0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
       0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025, 0.0025,
      

In [115]:
from sklearn.tree import DecisionTreeClassifier

In [116]:
T = 100 # Training rounds
h = [None] * T #[i for i in range(T)]
alpha = [None] * T
D_t = D_1
for t in range(T):
  h[t] = DecisionTreeClassifier(max_depth = 1)
  h[t].fit(X_train, y_train, sample_weight = D_t)
  y_pred = h[t].predict(X_train)

  eps_t = np.clip(np.sum(D_t * (y_pred != y_train)), 1e-10, 1 - 1e-10)
  if eps_t > 0.5: # early stopping
    break
  alpha[t] = 0.5 * np.log((1 - eps_t)/eps_t)
  Z = np.sum(D_t * np.exp(-alpha[t] * y_train * y_pred)) # Normalization factor
  D_t = D_t * np.exp(-alpha[t] * y_train * y_pred)/Z

lis = [None] * T
for i in range(T):
  lis[i] = alpha[i] * h[i].predict(X_test)
pred = np.sign(np.sum(lis, axis = 0))

max_depth = 1 to keep it a weak learner

otherwise the base learner is too powerful and next Dist is heavily skewed, leading subsequent learners to learn from very little left to correct.

In [117]:
acc = np.mean(pred == y_test)
acc

np.float64(0.91)